# Feature Engineering

This notebook is a **verbatim walkthrough** of feature engineering. It mirrors the production code but is isolated for safe explanation and debugging.

**Source file copied here:**
- nlp/feature_engineering.py

The code below is copied exactly as in the project.

## Verbatim Code: nlp/feature_engineering.py

In [ ]:
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from config.settings import VECTORIZER_PATH
from nlp.preprocessing import preprocess_text

def create_tfidf_features(corpus, save=True):
    """
    TF-IDF is a way to turn words into numbers.
    - 'TF' (Term Frequency): How many times a word appears.
    - 'IDF' (Inverse Document Frequency): How unique a word is across all documents.
    
    This helps the AI focus on 'important' words like 'Indemnify' rather than 'The'.
    
    Note: corpus is expected to already be preprocessed (lowercased, lemmatized, etc.)
    """
    # ngram_range=(1,2) means we look at single words ("risk") 
    # AND pairs of words ("high risk").
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=5000  # We only take the top 5000 most 'meaningful' words.
    )
    
    # The 'fit' part creates the dictionary of words.
    # The 'transform' part turns the text into numbers (a matrix).
    # corpus is already preprocessed by batch_preprocess_texts() in train.py.
    X = vectorizer.fit_transform(corpus)
    
    # We save our vectorizer 'dictionary' so we can use it again later.
    if save:
        os.makedirs(os.path.dirname(VECTORIZER_PATH), exist_ok=True)
        joblib.dump(vectorizer, VECTORIZER_PATH)
        print(f"AI 'Dictionary' saved at {VECTORIZER_PATH}")
        
    return X, vectorizer

def load_vectorizer():
    """Returns the saved TF-IDF engine."""
    if os.path.exists(VECTORIZER_PATH):
        return joblib.load(VECTORIZER_PATH)
    return None

def transform_new_text(texts, vectorizer=None):
    """Turns new, unseen text into numbers using a pre-saved dictionary."""
    if vectorizer is None:
        vectorizer = load_vectorizer()
        if vectorizer is None:
            raise FileNotFoundError("AI Dictionary (vectorizer) not found!")
    
    processed_texts = [preprocess_text(t) for t in texts]
    return vectorizer.transform(processed_texts)

### What this does, line by line
- `TfidfVectorizer` builds a vocabulary and maps text into weighted vectors.
- `ngram_range=(1, 2)` captures both single words and short phrases.
- `max_features=5000` limits vocabulary size for speed and regularization.
- `joblib.dump()` saves the fitted vectorizer so training and inference use identical vocabularies.
- `transform_new_text()` applies preprocessing again to keep inference consistent with training.

### Why this is important
- It converts raw language into a numeric matrix that classic ML models can learn from.
- The saved vectorizer guarantees that inference uses the same feature space as training.

### Alternatives considered (and why not used here)
- **CountVectorizer**: simpler but less informative; TF-IDF down-weights common legal filler.
- **Word2Vec / GloVe**: dense embeddings, but less transparent and harder to interpret for legal review.
- **BERT embeddings**: highly accurate but expensive to compute and harder to explain to non-ML audiences.